# Tutorial: Randomized Benchmarking with `errorgnomark`

Welcome to the `errorgnomark` tutorial! This notebook provides a step-by-step guide to using the framework's Randomized Benchmarking (RB) capabilities.

We will follow a logical progression, starting from generating a single abstract circuit and culminating in a complete, end-to-end Interleaved Randomized Benchmarking (IRB) experiment with data analysis and plotting.

### Learning Objectives:

*   **Part 1: Generate & Visualize a Logical RB Circuit:** Understand the high-level structure of an RB sequence.
*   **Part 2: Decompose to Physical Gates:** Learn how to translate abstract circuits into hardware-executable instructions.
*   **Part 3: Set Up an Interleaved RB (IRB) Experiment:** Isolate and measure the fidelity of a specific gate (e.g., `CZ`).
*   **Part 4: Analyze & Plot Existing Data:** Use the framework's tools to fit and visualize pre-acquired RB data.
*   **Part 5: Run an End-to-End Experiment:** Execute a full workflow, from circuit generation to simulation, analysis, and final plotting.

## Setup: Imports and Backend

First, we import the necessary components from the `errorgnomark` library and other tools like `numpy` and `matplotlib`.

The `try...except` block is designed to allow this notebook to run in two scenarios:
1.  When `errorgnomark` is properly installed as a package.
2.  When running directly from the source code repository (for development).

This tutorial assumes that a `dummy_backend` object, which simulates a noisy quantum device, has already been defined and is available in the environment. We will use this `dummy_backend` to run our circuits.

In [6]:
# --- Standard Library Imports ---
import numpy as np
import logging
from typing import List, Dict, Any, Union
import matplotlib.pyplot as plt

# --- Simplified Framework Imports ---
# NOTE: For this simple import to work, your project must be correctly 
# installed in your Python environment. See the instructions below.

from errorgnomark.circuits.circuit import QuantumCircuit,Gate
from errorgnomark.experiments.benchmarking.rb import StandardRBExperiment, InterleavedRBExperiment
from errorgnomark.analysis.rb import fit_rb_data,calculate_epg,analyze_epg,plot_rb_single,plot_rb_comparison

# Configure logging
logging.basicConfig(level=logging.INFO)

print("Successfully imported 'errorgnomark' modules.")

Successfully imported 'errorgnomark' modules.


## Part 1: Generate a Logical RB Circuit

Let's start by creating a `StandardRBExperiment` for two qubits. We can then ask it to generate a single logical (or abstract) RB circuit of a specific depth. This circuit consists of abstract Clifford group elements and does not yet specify the exact gate sequence.

In [8]:
# 1. Define the experiment parameters
qubits = [0, 1]
rb_exp_logical = StandardRBExperiment(qubits=qubits, seed=42)

# 2. Generate a single logical circuit with depth m=5
# --- FIX ---
# The 'decompose' argument is not part of the function's definition.
# Decomposition is controlled by whether 'native_gates' is set when creating the StandardRBExperiment object.
# Since we did not set it, the circuit will correctly remain logical (not decomposed).
logical_circuit = rb_exp_logical.generate_single_circuit(depth=5) # REMOVED: decompose=False

# 3. Print and visualize the circuit
print(logical_circuit)
print("\n--- Circuit Visualization ---")
logical_circuit.draw()

[INFO] RB Experiment configured for LOGICAL view (`native_gates=None`).


QuantumCircuit(qubits=[0, 1], num_gates=88)

--- Circuit Visualization ---
q0:     ┤  X  ├┤ SDG ├───●───┤  S  ├┤  H  ├┤  H  ├┤ SDG ├───●───┤  Y  ├┤  S  ├┤  S  ├┤  H  ├──────────●───┤  Y  ├┤  H  ├┤ SDG ├┤  H  ├┤ SDG ├┤  H  ├───●───┤  H  ├┤  S  ├──────────●───┤ SDG ├┤  S  ├─────────────────●───┤ SDG ├┤  H  ├──────────●───┤  H  ├┤  S  ├┤  H  ├┤  S  ├┤  H  ├┤  Y  ├───●───┤  H  ├┤ SDG ├┤ SDG ├┤  Y  ├──────────●───┤  S  ├┤  H  ├┤  H  ├┤ SDG ├───●───┤  S  ├┤  X  ├── M ──
q1:     ┤  X  ├┤  S  ├───⊕───┤ SDG ├┤  H  ├┤  Y  ├┤  H  ├───⊕───┤  Y  ├┤ SDG ├┤  H  ├┤ SDG ├┤  H  ├───⊕───┤ SDG ├┤  H  ├┤  S  ├────────────────────────⊕───┤  H  ├┤  Y  ├┤  X  ├───⊕───┤  H  ├┤ SDG ├┤  S  ├┤  H  ├───⊕───┤  X  ├┤  Y  ├┤  H  ├───⊕───┤ SDG ├┤  H  ├┤  S  ├────────────────────────⊕───┤  H  ├┤  S  ├┤  H  ├┤  S  ├┤  Y  ├───⊕───┤  H  ├┤  Y  ├┤  H  ├┤  S  ├───⊕───┤ SDG ├┤  X  ├── M ──


## Part 2: Decompose to Physical Gates

Real hardware cannot execute abstract 'Clifford' gates. They must be *decomposed* into a set of basis gates that the hardware supports, known as `native_gates`.

Let's create a new experiment, this time specifying a native gate set. When we generate a circuit now, it will be automatically decomposed.

In [9]:
# Assume 'qubits' is defined from the previous cell, e.g., qubits = [0, 1]

# 1. Define the native gate set for our hardware
native_gates = ['sx', 'rz', 'cz']

# 2. Create an experiment instance with the native gate set
# By providing 'native_gates' here, you are instructing the experiment
# to automatically decompose all circuits it generates into this basis.
rb_exp_physical = StandardRBExperiment(qubits=qubits, native_gates=native_gates, seed=42)

# 3. Generate a circuit of depth m=2. It will be automatically decomposed.
# We use a smaller depth because decomposition results in many more gates.

# --- FIX ---
# Removed the 'decompose=True' argument. The decomposition happens automatically
# because 'native_gates' was specified when creating the 'rb_exp_physical' object.
physical_circuit = rb_exp_physical.generate_single_circuit(depth=2)

# 4. Print and visualize the decomposed circuit
print(f"Circuit contains {len(physical_circuit.gates)} native gates.")
print("\n--- Decomposed Circuit Visualization (first 20 gates) ---")
# Drawing the full circuit can be very long, so we can't show it here.
# In a real scenario with the library's drawer, it would be displayed.
for gate in physical_circuit.gates[:20]:
    print(gate)

[INFO] RB Experiment configured for PHYSICAL view. Decomposing to: ['sx', 'rz', 'cz']


Circuit contains 126 native gates.

--- Decomposed Circuit Visualization (first 20 gates) ---
Gate(name='rz', qubits=(0,), params=(3.141592653589793,))
Gate(name='sx', qubits=(0,))
Gate(name='rz', qubits=(0,), params=(1.5707963267948966,))
Gate(name='sx', qubits=(0,))
Gate(name='rz', qubits=(0,), params=(3.141592653589793,))
Gate(name='sx', qubits=(0,))
Gate(name='rz', qubits=(0,), params=(1.5707963267948966,))
Gate(name='sx', qubits=(0,))
Gate(name='rz', qubits=(0,), params=(1.5707963267948966,))
Gate(name='sx', qubits=(1,))
Gate(name='rz', qubits=(1,), params=(1.5707963267948966,))
Gate(name='sx', qubits=(1,))
Gate(name='rz', qubits=(1,), params=(1.5707963267948966,))
Gate(name='sx', qubits=(1,))
Gate(name='rz', qubits=(1,), params=(1.5707963267948966,))
Gate(name='sx', qubits=(1,))
Gate(name='cz', qubits=(0, 1))
Gate(name='sx', qubits=(1,))
Gate(name='rz', qubits=(1,), params=(1.5707963267948966,))
Gate(name='sx', qubits=(1,))


## Part 3: Set Up an Interleaved RB (IRB) Experiment

Interleaved Randomized Benchmarking (IRB) is used to measure the fidelity of a specific gate. It works by sandwiching (interleaving) the target gate between each random Clifford in the sequence.

Here, we'll set up an IRB experiment to characterize the `CZ` gate.

In [ ]:
# Make sure necessary components are imported
from qiskit.circuit import QuantumCircuit
from qiskit_experiments.library import InterleavedRBExperiment
# We assume 'qubits' is defined, for example:
qubits = [0, 1]

# 1. Define the gate we want to benchmark as a QuantumCircuit
# The InterleavedRBExperiment expects an Instruction or QuantumCircuit.
# Creating a QuantumCircuit is the most robust method.
num_qubits_interleaved = 2
interleaved_element = QuantumCircuit(num_qubits_interleaved, name="cz")
interleaved_element.cz(0, 1)

# 2. Create an InterleavedRBExperiment instance
# --- FIX ---
# a) The argument is 'interleaved_element', not 'interleaved_gate'.
# b) To generate a LOGICAL circuit, we must OMIT the 'native_gates' argument.
#    The presence of 'native_gates' instructs the experiment to always output
#    decomposed (physical) circuits.
irb_exp = InterleavedRBExperiment(
    qubits=qubits, 
    interleaved_element=interleaved_element,
    seed=123
)

# 3. Generate a single logical IRB circuit to see the structure
# --- FIX ---
# Removed the 'decompose=False' argument as it is not a valid parameter.
# Because we created 'irb_exp' without 'native_gates', this will correctly
# generate a logical circuit.
irb_logical_circuit = irb_exp.generate_single_circuit(depth=3)

# 4. Visualize the logical IRB circuit
print("--- Logical IRB Circuit Visualization ---")
# The .draw() method is correct and will work on the generated circuit.
irb_logical_circuit

## Part 4: Analyze & Plot Existing Data

The `errorgnomark` library includes powerful tools for analyzing and plotting RB data. Let's imagine we have already run an experiment and collected the following survival probabilities.

We can use `fit_rb_data` to fit the exponential decay curve and `plot_rb_single` to visualize the result.

In [ ]:
# 1. Some pre-collected dummy data (e.g., from a previous experiment)
# This dictionary maps sequence depth (m) to a list of survival probabilities from different random circuits.
dummy_data = {
    1: [0.98, 0.99, 0.97], 
    10: [0.91, 0.92, 0.90], 
    20: [0.85, 0.86, 0.84],
    40: [0.74, 0.75, 0.76],
    80: [0.60, 0.61, 0.59]
}

# 2. Use the analysis function to fit the data
analysis_results = fit_rb_data(dummy_data, num_qubits=len(qubits))

print("--- Analysis Results ---")
print(f"Fit Parameter 'p': {analysis_results['fit_params']['p']:.4f} ± {analysis_results['fit_errs']['p_err']:.4f}")
print(f"EPC (Error Per Clifford): {analysis_results['epc']:.5f} ± {analysis_results['epc_err']:.5f}")

# 3. Use the plotting function to visualize the data and the fit
fig, ax = plt.subplots(figsize=(8, 5))
plot_rb_single(
    ax=ax,
    data=dummy_data, 
    fit_results=analysis_results,
    label='Standard RB',
    color='blue'
)
ax.set_title("RB Data and Exponential Fit")
plt.show()

## Part 5: Run an End-to-End Experiment

Now, let's tie everything together. We will perform a complete RB and IRB experiment workflow:

1.  **Define Experiments**: Set up both `StandardRBExperiment` and `InterleavedRBExperiment`.
2.  **Generate & Run Circuits**: Create a full set of circuits for various depths and run them on the backend.
3.  **Process and Analyze Data**: Fit the results for both experiments.
4.  **Calculate Gate Error**: Use the EPC from both fits to find the error of the interleaved gate (`CZ`).
5.  **Plot Comparison**: Visualize both decay curves on the same plot.

In [ ]:
# --- Main script for the end-to-end workflow ---

# 1. Define experiment parameters
qubits = [0, 1]
native_gates = ['sx', 'rz', 'cz']
gate_to_interleave = Gate(name='cz', qubits=[0, 1])
depths = [1, 10, 20, 40, 60, 80, 100]
num_circuits_per_depth = 20
shots = 1024

# 2. Set up both standard and interleaved experiments
std_rb_setup = StandardRBExperiment(qubits=qubits, native_gates=native_gates)
irb_setup = InterleavedRBExperiment(qubits=qubits, interleaved_gate=gate_to_interleave, native_gates=native_gates)

# Ensure 'dummy_backend' is defined in your environment before running this cell.

# 3. Run Standard RB Experiment
print("--- Starting Standard RB Experiment ---")
# 3.1 Generate all standard RB circuits
all_std_circuits = []
for depth in depths:
    for i in range(num_circuits_per_depth):
        std_rb_setup.seed = i
        circ = std_rb_setup.generate_single_circuit(depth=depth, decompose=True)
        all_std_circuits.append(circ)

# 3.2 Run circuits on the backend
print(f"Running {len(all_std_circuits)} standard circuits on the backend...")
std_raw_results = dummy_backend.run(all_std_circuits, shots=shots)

# 3.3 Process results into the analysis format
std_data = {depth: [] for depth in depths}
result_idx = 0
ground_state_str = '0' * len(std_rb_setup.qubits)
for depth in depths:
    for _ in range(num_circuits_per_depth):
        survival_prob = std_raw_results[result_idx].get(ground_state_str, 0.0)
        std_data[depth].append(survival_prob)
        result_idx += 1

# 4. Run Interleaved RB Experiment
print("\n--- Starting Interleaved RB Experiment ---")
# 4.1 Generate all interleaved RB circuits
all_irb_circuits = []
for depth in depths:
    for i in range(num_circuits_per_depth):
        irb_setup.seed = i
        circ = irb_setup.generate_single_circuit(depth=depth, decompose=True)
        all_irb_circuits.append(circ)

# 4.2 Run circuits on the backend
print(f"Running {len(all_irb_circuits)} interleaved circuits on the backend...")
irb_raw_results = dummy_backend.run(all_irb_circuits, shots=shots)

# 4.3 Process results into the analysis format
irb_data = {depth: [] for depth in depths}
result_idx = 0
# ground_state_str is the same, no need to redefine
for depth in depths:
    for _ in range(num_circuits_per_depth):
        survival_prob = irb_raw_results[result_idx].get(ground_state_str, 0.0)
        irb_data[depth].append(survival_prob)
        result_idx += 1

# 5. Analyze the data
print("\n--- Analyzing Data ---")
std_results = fit_rb_data(std_data, num_qubits=len(qubits))
irb_results = fit_rb_data(irb_data, num_qubits=len(qubits))

epc_std = std_results['epc']
epc_irb = irb_results['epc']

# 6. Calculate the interleaved gate error (EPG - Error Per Gate)
epg_cz_results = calculate_epg_from_rb(std_results, irb_results, num_qubits=len(qubits))
epg_cz = epg_cz_results['epg']
epg_cz_err = epg_cz_results['epg_err']

print(f"Standard EPC: {epc_std:.5f}")
print(f"Interleaved EPC: {epc_irb:.5f}")
print(f"Calculated EPG for CZ gate: {epg_cz:.5f} ± {epg_cz_err:.5f}")
print(f"CZ Gate Fidelity: {(1 - epg_cz):.4f}")

# 7. Plot the comparison
fig, ax = plt.subplots(figsize=(10, 6))
plot_rb_comparison(
    ax=ax,
    data_dict={'Standard': std_data, 'Interleaved (CZ)': irb_data},
    fit_results_dict={'Standard': std_results, 'Interleaved (CZ)': irb_results}
)
ax.set_title('End-to-End Standard vs. Interleaved RB')
plt.show()